In [41]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd

In [48]:
dfg_ids = [
    {
        'name': 'Carl von Ossietzky Universität Oldenburg',
        'dfg_id': 10233, 
    },
    {
        'name': 'Hochschule für Musik, Theater und Medien Hannover',
        'dfg_id': 10246, 
    },
    {
        'name': 'Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth',
        'dfg_id': 10533, 
    },
    {
        'name': 'Universität Osnabrück',
        'dfg_id': 10244, 
    },
    {
        'name': 'Ostfalia Hochschule für angewandte Wissenschaften',
        'dfg_id': 10254, 
    },
    {
        'name': 'Gottfried Wilhelm Leibniz Universität Hannover',
        'dfg_id': 10238, 
    },
    {
        'name': 'Hochschule Hannover',
        'dfg_id': 10252, 
    },
    {
        'name': 'Medizinische Hochschule Hannover (MHH)',
        'dfg_id': 10247, 
    },
    {
        'name': 'Georg-August-Universität Göttingen',
        'dfg_id': 10236, 
    },
    {
        'name': 'Technische Universität Braunschweig',
        'dfg_id': 10240, 
    },
    {
        'name': 'Hochschule Emden/Leer',
        'dfg_id': 980710, 
    },
    {
        'name': 'Stiftung Universität Hildesheim',
        'dfg_id': 10235, 
    },
    {
        'name': 'Stiftung Tierärztliche Hochschule Hannover',
        'dfg_id': 10249, 
    },
    {
        'name': 'Technische Universität Clausthal',
        'dfg_id': 10242, 
    },
    {
        'name': 'Hochschule Osnabrück',
        'dfg_id': 10255, 
    },
    {
        'name': 'Hochschule für Bildende Künste Braunschweig',
        'dfg_id': 10251, 
    },
    {
        'name': 'Universität Vechta',
        'dfg_id': 10597, 
    },
    {
        'name': 'Leuphana Universität Lüneburg',
        'dfg_id': 10232, 
    },
    {
        'name': 'HAWK Hochschule für angewandte Wissenschaft und Kunst',
        'dfg_id': 10253, 
    },
]

In [66]:
driver = webdriver.Firefox()
driver.get('https://gerit.org/de/institutiondetail/10236')

button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, '#associatedData span'))
)

# manchmal taucht ein cookie banner auf
driver.execute_script('arguments[0].click();', button)

element = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.ID, 'associated'))
)

content = driver.page_source
driver.quit()

In [67]:
soup = BeautifulSoup(content, features='html.parser')

In [68]:
list_of_institutions = soup.find(name='div', attrs={'id': 'associated'})

In [69]:
li = list_of_institutions.find_all(name='li')

In [132]:
for inst in li:
    list_of_spans = inst.find_all('span')
    #print(inst.find('span').text + ': ' + inst.find('a', href=True).get('href').replace('/de/institutiondetail/', '') + ' ' + (inst.find('div').get('class')[-1]))
    list_of_subs = inst.find_all(name='span', attrs={'class': 'associated-institution'})
    print(list_of_subs[0].text + ' ' + inst.find('div').get('class')[-1] + ': ' + inst.find('a', href=True).get('href').replace('/de/institutiondetail/', ''))
    print('#####')
    for i, subs in enumerate(list_of_subs):
        if i == 0:
            pass
        else:
            print(subs.text + ' ' + inst.find('div').get('class')[0])
    print('----------------')

Campus Institut für Dynamik biologischer Netzwerke (CIDBN) level0: 443884006
#####
Abteilung Datengetriebene Analyse biologischer Netzwerke associated-contend
----------------
Abteilung Datengetriebene Analyse biologischer Netzwerke level1: 443885740
#####
----------------
Campus-Institut Data Science (CIDAS) level0: 565050847
#####
----------------
Centre for Modern Indian Studies (CeMIS) level0: 179419375
#####
Forschungsgruppe Moderne Indische Geschichte associated-contend
----------------
Forschungsgruppe Moderne Indische Geschichte level1: 517973470
#####
----------------
Courant Forschungszentrum "Armut, Ungleichheit und Wachstum in Entwicklungsländern" level0: 232350829
#####
----------------
Courant Forschungszentrum "Bildung und Religion" level0: 214341364
#####
Area 1: Pietas und Paideia. Religiöse Traditionen und intellektuelle Kulturen in der Welt des Römischen Reiches associated-contend
----------------
Area 1: Pietas und Paideia. Religiöse Traditionen und intellektuelle K

In [56]:
df = pd.DataFrame.from_dict(dict(dfg_id=[], inst_name=[], level=[], parent_name=[], dfg_id_parent=[]))

for dfg_id_dict in dfg_ids:

    dfg_id = dfg_id_dict.get('dfg_id')
    parent_name = dfg_id_dict.get('name')
    
    driver = webdriver.Firefox()
    driver.get(f'https://gerit.org/de/institutiondetail/{dfg_id}')
    
    button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, '#associatedData span'))
    )

    # manchmal taucht ein cookie banner auf
    driver.execute_script('arguments[0].click();', button)
    
    element = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, 'associated'))
    )
    
    content = driver.page_source
    driver.quit()

    soup = BeautifulSoup(content, features='html.parser')
    list_of_institutions = soup.find(name='div', attrs={'id': 'associated'})
    li = list_of_institutions.find_all(name='li')

    inst_dict = dict(
        dfg_id = [],
        inst_name = [],
        level = [],
        parent_name = [],
        dfg_id_parent = []
    )

    for inst in li:
        inst_dict['dfg_id'].append(inst.find('a', href=True).get('href').replace('/de/institutiondetail/', ''))
        inst_dict['level'].append(inst.find('div').get('class')[-1])
        inst_dict['inst_name'].append(inst.find('span').text)
        inst_dict['parent_name'].append(parent_name)
        inst_dict['dfg_id_parent'].append(dfg_id)

    df2 = pd.DataFrame.from_dict(inst_dict)

    df = pd.concat([df, df2], ignore_index=True)

    df['dfg_id'] = df['dfg_id'].astype(int)
    df['dfg_id_parent'] = df['dfg_id_parent'].astype(int)

In [57]:
df

,dfg_id,inst_name,level,parent_name,dfg_id_parent
0,14332,Bibliotheks- und Informationssystem (BIS),level0,Carl von Ossietzky Universität Oldenburg,10233
1,14349,Center für lebenslanges Lernen (C3L),level0,Carl von Ossietzky Universität Oldenburg,10233
2,19052,"CMC - Center for Migration, Education and Cult...",level0,Carl von Ossietzky Universität Oldenburg,10233
3,35245680,COAST - Zentrum für Umwelt- und Nachhaltigkeit...,level0,Carl von Ossietzky Universität Oldenburg,10233
4,25628,ecco-ecology + communication Unternehmensberat...,level0,Carl von Ossietzky Universität Oldenburg,10233
...,...,...,...,...,...
2075,9409718,Fakultät Gestaltung,level1,HAWK Hochschule für angewandte Wissenschaft un...,10253
2076,21419305,Fakultät soziale Arbeit und Gesundheit,level1,HAWK Hochschule für angewandte Wissenschaft un...,10253
2077,277626399,"Studiengang MSc Ergotherapie, Logopädie, Physi...",level2,HAWK Hochschule für angewandte Wissenschaft un...,10253
2078,133037425,Standort Holzminden,level0,HAWK Hochschule für angewandte Wissenschaft un...,10253


In [58]:
df.to_csv('../data/gerit_inst_with_ids.csv', index=False)

In [87]:
df[df.dfg_id_parent == 10236].head(20)

,dfg_id,inst_name,level,parent_name,dfg_id_parent
909,443884006,Campus Institut für Dynamik biologischer Netzw...,level0,Georg-August-Universität Göttingen,10236
910,443885740,Abteilung Datengetriebene Analyse biologischer...,level1,Georg-August-Universität Göttingen,10236
911,565050847,Campus-Institut Data Science (CIDAS),level0,Georg-August-Universität Göttingen,10236
912,179419375,Centre for Modern Indian Studies (CeMIS),level0,Georg-August-Universität Göttingen,10236
913,517973470,Forschungsgruppe Moderne Indische Geschichte,level1,Georg-August-Universität Göttingen,10236
914,232350829,"Courant Forschungszentrum ""Armut, Ungleichheit...",level0,Georg-August-Universität Göttingen,10236
915,214341364,"Courant Forschungszentrum ""Bildung und Religion""",level0,Georg-August-Universität Göttingen,10236
916,252506619,Area 1: Pietas und Paideia. Religiöse Traditio...,level1,Georg-August-Universität Göttingen,10236
917,171578186,"Courant Forschungszentrum ""Evolution des Sozia...",level0,Georg-August-Universität Göttingen,10236
918,102731477,"Courant Forschungszentrum ""Nanospektroskopie u...",level0,Georg-August-Universität Göttingen,10236
